# Surrogate Factory — UCLoads
## Chapter 7. Model Training
Objectives:
- Train MLP and GradientBoosting on the scaled training set.
- Log training metrics (loss curves) to MLflow.
- Save model artifacts (`.modl` files).

### 0. Workflow initialisation

In [ ]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()

### 7. Model Training

In [ ]:
workflow.import_metadata(stage_name="SF_7_Model_Training")

In [ ]:
job = workflow.config['job_name']
Train_set = workflow.load_data(job + '_Train_set.csv')
Val_set   = workflow.load_data(job + '_Val_set.csv')
print(f"Train: {Train_set.shape}  Val: {Val_set.shape}")

In [ ]:
from model_training.learn import train
models_info = train(workflow, Train_set, Val_set)
print(f"\nTrained {len(models_info)} models.")

#### 7.1 MLP training curves

In [ ]:
%matplotlib inline
import joblib, matplotlib.pyplot as plt

for info in models_info:
    model = joblib.load(info['file'])
    if hasattr(model, 'loss_curve_'):
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(model.loss_curve_, label='Train loss')
        if hasattr(model, 'validation_scores_') and model.validation_scores_:
            ax.plot(model.validation_scores_, label='Val score', linestyle='--')
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Loss / Score')
        ax.set_title(f"{info['label']} — Training Curve")
        ax.legend()
        plt.tight_layout()
        plt.show()
        plt.close()

### Save

In [ ]:
workflow.save_metadata()